# PANOSETI Topology Engine Demo

This notebook demonstrates how to use the PANOSETI topology engine to model, visualize, and validate observatory fleet configurations.

In [ ]:
import os
import sys
import networkx as nx
import matplotlib.pyplot as plt

# Add src to path if running from within the control directory
sys.path.insert(0, os.path.abspath("src"))

from control.topology.fleet import generate_fleet_configs
from control.topology.graph_builder import GraphBuilder
from control.topology.visualizer import save_topology_image, export_topology_json
from control.utils.global_validator import GlobalConfigValidator

## 1. Programmatic Fleet Generation

We can create an arbitrary $n$-node fleet with randomized subnets.

In [ ]:
# Create a 4-node fleet, each managing 1 module, 50% chance of subnets
daq_config, quabo_uids = generate_fleet_configs(
    num_daq_nodes=4, 
    modules_per_node=1, 
    subnet_probability=0.5
)

print(f"Generated fleet with {len(daq_config.daq_nodes)} DAQ nodes.")
for node in daq_config.daq_nodes:
    status = "behind Gateway" if node.port_forwarding and node.port_forwarding.status else "Direct"
    print(f" - Node {node.ip_addr}: {status}")

## 2. Building the Topology Graph

The `GraphBuilder` converts these configurations into a NetworkX Directed Graph (DiGraph).

In [ ]:
builder = GraphBuilder()
graph = builder.build_from_configs(daq_config, quabo_uids)

print(f"Graph built with {graph.number_of_nodes()} nodes and {graph.number_of_edges()} edges.")
print("Node Roles:", set(nx.get_node_attributes(graph, 'role').values()))

## 3. Visualization

We can render the graph using Matplotlib.

In [ ]:
%matplotlib inline
plt.figure(figsize=(12, 8))

# Define layout
pos = nx.spring_layout(graph, k=0.5, iterations=50)

# Color map
colors = {
    "headnode": "red", 
    "daqnode": "blue", 
    "gateway": "green", 
    "module": "orange", 
    "quabo": "skyblue"
}
node_colors = [colors.get(graph.nodes[n].get('role'), 'gray') for n in graph.nodes]

nx.draw(graph, pos, with_labels=True, node_color=node_colors, node_size=500, font_size=8)
plt.title("Live Topology Graph")
plt.show()

## 4. Structural Validation

Run the global validator to check for bottlenecks and reachability errors.

In [ ]:
configs = {
    'daq': daq_config,
    'obs': MagicMock(), # We only need daq/uids for topology checks
    'data': None,
    'network': None,
    'firmware': None
}

from unittest.mock import MagicMock
validator = GlobalConfigValidator(configs)
validator._check_topology_structural_integrity()

validator.report.print_report()

## 5. Exporting Data

Export to JSON for Cytoscape or D3.js.

In [ ]:
export_topology_json(graph, "topology_export.json")
print("Exported to topology_export.json")